# EuroCropML Benchmark - Step 3: Classical Baselines
Load pre-extracted features from Drive and train RF/LGBM/XGB classifiers.

In [ ]:
!git clone https://github.com/mahdikalantari555/eurocrop-olmoearth-benchmark.git
%cd eurocrop-olmoearth-benchmark
!pip install -r requirements.txt -q

In [ ]:
import json, os, time, gc
import numpy as np
from src.data.prep_features import load_prepared_features
from src.models.classical import get_classifier
from src.evaluate.metrics import compute_metrics, save_metrics, save_confusion_matrix

In [ ]:
# Mount Drive and load features
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/eurocrop_benchmark'
os.makedirs('results/metrics', exist_ok=True)

In [ ]:
# NDVI + Random Forest
X_tr, y_tr, X_te, y_te, labels = load_prepared_features(os.path.join(DRIVE_DIR, 'features_ndvi'))
print(f"NDVI features: train={X_tr.shape}, test={X_te.shape}")

t = time.time()
clf = get_classifier("rf", 42)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

m = compute_metrics(y_te, y_pred, labels=labels)
save_metrics(m, 'results/metrics/phase1_ndvi_rf.json')
save_confusion_matrix(y_te, y_pred, 'results/metrics/phase1_ndvi_rf_cm.csv', labels=labels)
print(f"ndvi_rf: OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} ({time.time()-t:.1f}s)")

del X_tr, clf
gc.collect()

In [ ]:
# BandStat + RF/LGBM/XGB
X_tr, y_tr, X_te, y_te, labels = load_prepared_features(os.path.join(DRIVE_DIR, 'features_bandstat'))
print(f"BandStat features: train={X_tr.shape}, test={X_te.shape}")

for clf_name in ["rf", "lgbm", "xgb"]:
    t = time.time()
    clf = get_classifier(clf_name, 42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    m = compute_metrics(y_te, y_pred, labels=labels)
    save_metrics(m, f'results/metrics/phase1_bandstat_{clf_name}.json')
    save_confusion_matrix(y_te, y_pred, f'results/metrics/phase1_bandstat_{clf_name}_cm.csv', labels=labels)
    print(f"bandstat_{clf_name}: OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} ({time.time()-t:.1f}s)")

    del clf
    gc.collect()

In [ ]:
# Copy results to Drive
!cp results/metrics/phase1_*.json /content/drive/MyDrive/eurocrop_benchmark/
print("Results saved to Drive")